# 03b · Variables externas y patrones para el mapa

> **Reto 2 del INE:** "¿Qué zonas podrían presentar mayor abstención en el futuro si no se interviene?"

**A qué responde este paso.** El notebook 03 dejó la historia limpia de cada municipio. Aquí respondemos tres preguntas:

1. **¿Qué otros datos públicos ayudan a anticipar dónde se votará menos?** Usamos el INEGI, la CONAPO, el CONEVAL, Banxico, el SESNSP y las actas del INE.
2. **¿Qué patrones debe ver el INE en el mapa?** Edad, generaciones, territorio, falta de casillas y credencialización.
3. **¿Con qué variables entra cada municipio al modelo?** Armamos una tabla por transición presidencial → intermedia, con **solo información disponible antes de cada elección**.

Los resultados, con su fuente y qué tan firmes son, están en `docs/investigacion/J-variables-externas.md`. Los indicadores para el mapa, en `docs/investigacion/K-patrones-y-alertas.md`. Las fuentes y sus verificaciones, en `docs/fuentes-externas.md`.

## Resumen de lo que encontramos

1. **La edad y el sexo explican menos del 1 % de las diferencias entre municipios.** Lo que pesa es el lugar. La edad sirve para decidir **a quién** dirigir la campaña, no **dónde**.
2. **El votante primerizo se pierde.** A los 18 años se vota mucho; en la intermedia siguiente, esa generación cae más que el resto de su municipio en 9 de cada 10 municipios.
3. **Faltan jóvenes en la lista.** En el municipio típico, el 31 % de la generación de 18 y 19 años no tiene credencial (2021).
4. **La abstención se agrupa en regiones:** 371 municipios forman zonas frías en 2024, la mayoría en Oaxaca.
5. **Elegir ayuntamiento el mismo día sube la participación federal.** Dentro de Oaxaca, la brecha entre municipios de partidos y de usos y costumbres pasó de 2–8 puntos a 12–21 cuando los de partidos empezaron a elegir ayuntamiento ese día.
6. **Lo que anticipa la intermedia en las dos transiciones pasadas (2015 y 2021):** elegir ayuntamiento o gobernador el mismo día, y en menor medida % de hombres, cobertura de la lista, usos y costumbres, escolaridad y lo rural. La migración, la pobreza y la competitividad electoral **no** se sostienen en las dos.
7. **Varias hipótesis no se sostuvieron:**
   - la pobreza no baja la participación;
   - la violencia municipal no anticipa dónde faltan casillas;
   - la migración explica el nivel, pero no mejora la predicción;
   - el margen de la elección presidencial anticipó 2021, pero no 2015.
8. **No tener policía municipal no va con votar menos.** Según el INEGI, 517 municipios no tenían policía propia en 2024. Fuera de los de usos y costumbres, votan igual que los que sí tienen. Lo que separa a los 400 municipios con cuerpos comunales es que casi todos son de usos y costumbres en Oaxaca.
9. **El dato de violencia no es igual de completo en todos los estados.** El INEGI cuenta más homicidios que el SESNSP en Guanajuato, Yucatán, Hidalgo, Estado de México y Tlaxcala; en Veracruz, Oaxaca, Jalisco, Coahuila y Durango pasa lo contrario. Por eso proponemos que la capa de violencia del mapa use los homicidios del INEGI.

## Palabras que usamos

| Palabra | Qué significa |
|---|---|
| **Desviación** | Participación del municipio menos la nacional, en puntos porcentuales (pp). |
| **Residuo o "lo que la historia no anticipa"** | Diferencia entre la desviación real en la intermedia y la que se esperaba por la presidencial anterior. Una variable útil para el modelo debe relacionarse con esto. |
| **Correlación de Spearman (ρ)** | Qué tanto se ordenan juntos dos datos, de −1 a 1. 0 = sin relación. Es robusta a valores extremos. |
| **IC 95 %** | Intervalo de confianza calculado con bootstrap: si no incluye el 0, la relación probablemente no es casual. |
| **Transición** | Una elección presidencial y la intermedia siguiente: 2012 → 2015, 2018 → 2021 y 2024 → 2027. |
| **Cobertura de la lista** | Lista nominal entre población del mismo grupo de edad y sexo. Puede pasar de 100 % si en la lista siguen personas que ya no viven ahí. |

In [1]:
from pathlib import Path
import io
import warnings

import numpy as np
import polars as pl
from scipy.stats import rankdata, spearmanr, mannwhitneyu

warnings.filterwarnings('ignore')
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(14)
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_width_chars(190)

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() and (p / 'data').is_dir())
E = raiz / 'data/processed/externos'
requeridos = ['panel_municipal.parquet', 'concurrencia_municipal.parquet', 'secciones_territorio_2024.parquet']
faltan = [f for f in requeridos if not (raiz / 'data/processed' / f).exists()]
assert not faltan, f'Primero corre el notebook 03; faltan {faltan}'
externos = ['inegi_municipal_2010', 'inegi_municipal_2020', 'inegi_edad_desplegada', 'conapo_proyecciones_edad_sexo', 'conapo_marginacion',
            'conapo_intensidad_migratoria', 'coneval_pobreza_municipal', 'inegi_homicidios_municipio', 'sesnsp_delitos_municipio', 'actas_2012',
            'cngmd_seguridad_municipal', 'envipe_cifra_oculta_estatal', 'sesnsp_victimas_estatal', 'inegi_homicidios_no_especificado']
faltan = [f for f in externos if not (E / f'{f}.parquet').exists()]
assert not faltan, f'Faltan fuentes externas {faltan}: corre los scripts de scripts/externos/ (ver docs/fuentes-externas.md)'

panel = pl.read_parquet(raiz / 'data/processed/panel_municipal.parquet')
concurrencia = pl.read_parquet(raiz / 'data/processed/concurrencia_municipal.parquet')
secciones = pl.read_parquet(raiz / 'data/processed/secciones_territorio_2024.parquet')
rng = np.random.default_rng(2027)

def spearman_ic(x, y, veces=1000):
    # Correlación de Spearman con intervalo de confianza de 95 % por bootstrap (rangos promedio en empates)
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = ~(np.isnan(x) | np.isnan(y)); x, y = x[ok], y[ok]
    rho = spearmanr(x, y).statistic
    bs = [spearmanr(x[i], y[i]).statistic for i in (rng.integers(0, len(x), len(x)) for _ in range(veces))]
    return rho, np.percentile(bs, 2.5), np.percentile(bs, 97.5), len(x)
print('Fuentes cargadas.')

Fuentes cargadas.


## 1 · ¿Quién vota? Edad y sexo

Armamos la lista y la participación por municipio (territorio de 2024), sexo y grupo de edad quinquenal, con las mismas reglas del notebook 03. Las secciones que no cuadran con el acta (D10) pasan a "no sabemos".

In [2]:
mapa_secciones = secciones.select('AELEC', 'EDOCVE', 'SECCION', 'municipio_2024', pl.col('no_cuadra_con_acta').fill_null(False))
grupo = (pl.when(pl.col('EDAD') < 20).then(pl.lit('18-19')).when(pl.col('EDAD') >= 85).then(pl.lit('85+'))
         .otherwise(((pl.col('EDAD') // 5) * 5).cast(pl.Utf8) + pl.lit('-') + ((pl.col('EDAD') // 5) * 5 + 4).cast(pl.Utf8)))
por_edad = (
    pl.scan_parquet(raiz / 'data/processed/ccpc/ccpc-*.parquet').filter(pl.col('SEXO') < 2)
    .with_columns(grupo_edad=grupo, sexo=pl.when(pl.col('SEXO') == 0).then(pl.lit('hombre')).otherwise(pl.lit('mujer')))
    .group_by('AELEC', 'EDOCVE', 'SECCION', 'sexo', 'grupo_edad').agg(pl.col('LN', 'SV', 'NV', 'NS').sum()).collect()
    .join(mapa_secciones, on=['AELEC', 'EDOCVE', 'SECCION'], how='left')
)
assert por_edad['municipio_2024'].null_count() == 0
d10 = pl.col('no_cuadra_con_acta')
por_edad = por_edad.with_columns(
    NS=pl.when(d10).then(pl.col('NS') + pl.col('SV') + pl.col('NV')).otherwise(pl.col('NS')),
    SV=pl.when(d10).then(0).otherwise(pl.col('SV')), NV=pl.when(d10).then(0).otherwise(pl.col('NV')),
).group_by('AELEC', pl.col('municipio_2024').alias('cvegeo'), 'sexo', 'grupo_edad').agg(pl.col('LN', 'SV', 'NV', 'NS').sum())
por_edad.write_parquet(raiz / 'data/processed/lista_edad_sexo_municipal.parquet')

curva = por_edad.group_by('AELEC', 'sexo', 'grupo_edad').agg(p=(pl.col('SV').sum() / (pl.col('SV').sum() + pl.col('NV').sum()) * 100).round(1))
curva.pivot(on='AELEC', index=['grupo_edad', 'sexo'], values='p', sort_columns=True).sort('grupo_edad', 'sexo').rename({'grupo_edad': 'Edad', 'sexo': 'Sexo'})

Edad,Sexo,2009,2012,2015,2018,2021,2024
str,str,f64,f64,f64,f64,f64,f64
"""18-19""","""hombre""",40.8,58.9,41.1,57.6,48.2,52.7
"""18-19""","""mujer""",47.2,65.2,47.3,63.6,55.1,60.8
"""20-24""","""hombre""",30.8,49.2,32.1,48.3,37.2,42.9
"""20-24""","""mujer""",39.7,58.3,40.4,57.4,46.3,53.9
"""25-29""","""hombre""",29.2,46.3,29.6,47.3,34.2,40.6
"""25-29""","""mujer""",39.5,58.3,40.2,58.4,45.1,53.9
"""30-34""","""hombre""",32.3,48.3,34.0,52.0,37.5,45.0
"""30-34""","""mujer""",42.9,61.0,46.0,64.0,48.9,58.7
"""35-39""","""hombre""",37.3,55.8,39.9,55.8,42.2,49.1


**Qué vemos** (participación nacional por grupo, en %):

- **Efecto "primera vez":** en todas las elecciones, las personas de 18 y 19 años votan más que las de 20 a 29.
- **El mínimo de la vida está entre los 25 y los 29 años.** En 2024, los hombres de esa edad votaron 40.6 %, el valor más bajo de todos los grupos.
- **Las mujeres votan más que los hombres hasta los 60 años.** La brecha más grande está entre los 25 y los 34 (unos 13 puntos en 2024). Desde los 65, votan más los hombres.
- **Los mayores votan cada vez más:** el grupo de 65 a 69 años pasó de 59 % en 2009 a 76.5 % en 2024.

### ¿Es la edad o la generación?

Seguimos a cada generación por su año de nacimiento, con la edad año por año.

In [3]:
gen = (pl.scan_parquet(raiz / 'data/processed/ccpc/ccpc-*.parquet').filter(pl.col('SEXO') < 2, pl.col('EDAD') <= 90)
       .group_by('AELEC', 'EDAD').agg(pl.col('SV', 'NV').sum()).collect()
       .with_columns(nacidos_en=pl.col('AELEC') - pl.col('EDAD'), p=(pl.col('SV') / (pl.col('SV') + pl.col('NV')) * 100).round(1)))
(gen.filter(pl.col('nacidos_en').is_in([2006, 2003, 2000, 1997, 1994, 1991, 1985, 1970, 1950]))
 .pivot(on='AELEC', index='nacidos_en', values='p', sort_columns=True).sort('nacidos_en', descending=True).rename({'nacidos_en': 'Nacidos en'}))

Nacidos en,2009,2012,2015,2018,2021,2024
i16,f64,f64,f64,f64,f64,f64
2006,null,null,null,null,null,61.5
2003,null,null,null,null,55.8,48.9
2000,null,null,null,64.8,42.4,47.1
1997,null,null,48.8,53.4,40.1,47.2
1994,null,66.7,36.7,52.1,39.6,50.3
1991,49.3,54.5,35.3,52.9,41.1,52.8
1985,33.9,52.4,37.2,59.2,47.2,57.8
1970,44.3,66.0,52.0,68.0,59.7,68.1
1950,56.9,74.3,62.9,73.3,68.3,74.2


**Qué vemos:**

- **A los 18 años se vota mucho, y tres años después se cae** entre 7 y 30 puntos, aunque la siguiente elección sea presidencial. Por ejemplo, los nacidos en 2003 votaron 55.8 % en 2021 y 48.9 % en 2024.
- **Después, cada generación sube despacio con la edad.**

¿Pasa en todos los municipios? Seguimos a la generación que tenía de 18 a 20 años en 2018 (nacida en 1998–2000) hasta 2021. Es la misma transición presidencial → intermedia que habrá de 2024 a 2027.

In [4]:
primerizos = (
    pl.scan_parquet(raiz / 'data/processed/ccpc/ccpc-*.parquet').filter(pl.col('SEXO') < 2, pl.col('AELEC').is_in([2018, 2021]))
    .with_columns(nacidos_en=pl.col('AELEC') - pl.col('EDAD')).filter(pl.col('nacidos_en').is_between(1998, 2000))
    .group_by('AELEC', 'EDOCVE', 'SECCION').agg(pl.col('SV', 'NV').sum()).collect()
    .join(mapa_secciones, on=['AELEC', 'EDOCVE', 'SECCION']).filter(~pl.col('no_cuadra_con_acta'))
    .group_by('AELEC', pl.col('municipio_2024').alias('cvegeo')).agg(pl.col('SV', 'NV').sum())
    .with_columns(p=pl.col('SV') / (pl.col('SV') + pl.col('NV')) * 100, base=pl.col('SV') + pl.col('NV'))
    .pivot(on='AELEC', index='cvegeo', values=['p', 'base'])
    .filter((pl.col('base_2018') >= 50) & (pl.col('base_2021') >= 50))
    .join(panel.filter(~pl.col('sin_casilla')).pivot(on='AELEC', index='cvegeo', values='participacion').select('cvegeo', total_2018='2018', total_2021='2021'), on='cvegeo')
    .with_columns(caida_generacion=pl.col('p_2021') - pl.col('p_2018'), caida_municipio=pl.col('total_2021') - pl.col('total_2018'))
    .with_columns(caida_extra=pl.col('caida_generacion') - pl.col('caida_municipio'))
)
primerizos.select(
    pl.len().alias('Municipios (con 50+ personas de la generación)'),
    pl.col('caida_generacion').median().round(1).alias('Caída de la generación (pp, mediana)'),
    pl.col('caida_municipio').median().round(1).alias('Caída del municipio (pp, mediana)'),
    pl.col('caida_extra').median().round(1).alias('Caída extra de los primerizos (pp, mediana)'),
    ((pl.col('caida_extra') < 0).mean() * 100).round(1).alias('% de municipios donde caen más que el total'),
)

Municipios (con 50+ personas de la generación),"Caída de la generación (pp, mediana)","Caída del municipio (pp, mediana)","Caída extra de los primerizos (pp, mediana)",% de municipios donde caen más que el total
u32,f64,f64,f64,f64
2225,-11.7,-5.1,-6.1,90.6


**Qué vemos:** en el municipio típico, la generación primeriza de 2018 cayó 11.7 puntos en 2021, **6.1 más que su municipio**, y cayó más que el total en el **90.6 % de los municipios**.

**Para 2027:** los primerizos de 2024 estarán en la misma situación. Es un grupo concreto y medible para la intervención: **retener al votante primerizo**.

### ¿La estructura por edad y sexo explica dónde se vota menos?

Para cada municipio calculamos la participación que tendría si cada grupo de edad y sexo votara como el promedio nacional de ese grupo (**efecto de la estructura**). La diferencia contra la real es el **efecto del lugar**.

In [5]:
tasas = por_edad.group_by('AELEC', 'sexo', 'grupo_edad').agg(p_grupo=pl.col('SV').sum() / (pl.col('SV').sum() + pl.col('NV').sum()))
esperada = (por_edad.join(tasas, on=['AELEC', 'sexo', 'grupo_edad']).with_columns(base=pl.col('SV') + pl.col('NV'))
            .group_by('AELEC', 'cvegeo').agg(esperada=(pl.col('base') * pl.col('p_grupo')).sum() / pl.col('base').sum() * 100))
efectos = (panel.filter(pl.col('fuente_participacion') == 'conteos censales').join(esperada, on=['AELEC', 'cvegeo'])
           .with_columns(estructura=pl.col('esperada') - pl.col('participacion_nacional'), lugar=pl.col('participacion') - pl.col('esperada')))
(efectos.filter(pl.col('AELEC') >= 2015).group_by('AELEC').agg(
    (pl.col('estructura').var() / pl.col('desviacion').var() * 100).round(1).alias('Varianza explicada por la estructura (%)'),
    pl.col('estructura').quantile(0.05).round(1).alias('Efecto de la estructura, percentil 5 (pp)'),
    pl.col('estructura').quantile(0.95).round(1).alias('Efecto de la estructura, percentil 95 (pp)'),
).sort('AELEC').rename({'AELEC': 'Elección'}))

Elección,Varianza explicada por la estructura (%),"Efecto de la estructura, percentil 5 (pp)","Efecto de la estructura, percentil 95 (pp)"
i16,f64,f64,f64
2015,0.4,-1.4,1.9
2018,0.4,-1.2,1.0
2021,0.5,-1.6,1.9
2024,1.0,-1.8,2.1


**Qué vemos:** la estructura por edad y sexo explica **alrededor del 1 % o menos** de la variación entre municipios, y su efecto va de unos −2 a +2 puntos. **Casi toda la diferencia es efecto del lugar.**

**Consecuencia para el modelo:** para predecir **dónde** se vota menos sirven las características del lugar, no la pirámide de edades.

## 2 · Jóvenes sin credencial

Comparamos a la **misma generación**: la lista nominal de junio de 2021 en cada edad contra la población del censo de marzo de 2020 con un año menos. El censo redondea edades (20, 30, 45…), así que por edad simple solo son confiables los 18 y 19 años.

In [6]:
lista_edad_2021 = (pl.scan_parquet(raiz / 'data/processed/ccpc/ccpc-2021.parquet').filter(pl.col('SEXO') < 2)
                   .with_columns(sexo=pl.when(pl.col('SEXO') == 0).then(pl.lit('hombre')).otherwise(pl.lit('mujer')))
                   .group_by('AELEC', 'EDOCVE', 'SECCION', 'sexo', 'EDAD').agg(pl.col('LN', 'SV', 'NV').sum()).collect()
                   .join(mapa_secciones.select('AELEC', 'EDOCVE', 'SECCION', 'municipio_2024'), on=['AELEC', 'EDOCVE', 'SECCION'])
                   .group_by(pl.col('municipio_2024').alias('cvegeo'), 'sexo', 'EDAD').agg(pl.col('LN', 'SV', 'NV').sum()))
censo = pl.read_parquet(E / 'inegi_edad_desplegada.parquet').filter(pl.col('anio') == 2020).select('cvegeo', 'sexo', pl.col('edad').alias('edad_censo'), 'poblacion')
misma_generacion = lista_edad_2021.with_columns(edad_censo=(pl.col('EDAD') - 1).cast(pl.Int16)).join(censo, on=['cvegeo', 'sexo', 'edad_censo'])
nacional = misma_generacion.group_by('EDAD').agg(cobertura=(pl.col('LN').sum() / pl.col('poblacion').sum() * 100).round(1))
credencial = (misma_generacion.filter(pl.col('EDAD').is_between(18, 19)).group_by('cvegeo').agg(pl.col('LN', 'poblacion', 'SV', 'NV').sum())
              .filter(pl.col('poblacion') >= 50)
              .with_columns(credencializacion_18_19=pl.col('LN') / pl.col('poblacion') * 100, participacion_18_19=pl.col('SV') / (pl.col('SV') + pl.col('NV')) * 100))
print(nacional.filter(pl.col('EDAD').is_in([18, 19, 22, 25])).sort('EDAD').rename({'EDAD': 'Edad en 2021', 'cobertura': 'En la lista / población de su generación (%)'}))
credencial.select(
    pl.len().alias('Municipios'),
    pl.col('credencializacion_18_19').median().round(1).alias('Credencialización 18–19 (mediana, %)'),
    pl.col('credencializacion_18_19').quantile(0.1).round(1).alias('Percentil 10 (%)'),
    pl.col('participacion_18_19').median().round(1).alias('Votó, de los que están en la lista (mediana, %)'),
    (pl.col('participacion_18_19') * pl.col('credencializacion_18_19').clip(upper_bound=100) / 100).median().round(1).alias('Votó, de toda su generación (mediana, %)'),
)

shape: (4, 2)
┌──────────────┬──────────────────────────────────────────────┐
│ Edad en 2021 ┆ En la lista / población de su generación (%) │
│ ---          ┆ ---                                          │
│ i16          ┆ f64                                          │
╞══════════════╪══════════════════════════════════════════════╡
│ 18           ┆ 59.5                                         │
│ 19           ┆ 82.4                                         │
│ 22           ┆ 114.7                                        │
│ 25           ┆ 110.4                                        │
└──────────────┴──────────────────────────────────────────────┘


Municipios,"Credencialización 18–19 (mediana, %)",Percentil 10 (%),"Votó, de los que están en la lista (mediana, %)","Votó, de toda su generación (mediana, %)"
u32,f64,f64,f64,f64
2232,68.6,48.3,63.1,41.0


**Qué vemos:**

- **Solo el 59.5 % de quienes tenían 18 años en 2021 estaba en la lista; a los 19, el 82.4 %.**
- **En el municipio típico, el 68.6 % de la generación de 18 y 19 años tiene credencial**; en el 10 % más bajo, menos del 48.3 %.
- **Votó el 63 % de los que estaban en la lista, pero solo el 41 % de toda su generación.**

**Para el INE:** antes de motivar el voto joven hay un paso propio del INE, **credencializar a quienes cumplirán 18 años antes de junio de 2027**. Es una vista del mapa (indicador K22).

## 3 · La abstención en el territorio

¿Los municipios que votan poco están juntos? Usamos las fronteras municipales del INEGI y consideramos vecinos a los municipios que comparten frontera o un vértice.

In [7]:
import geopandas as gpd
from libpysal.weights import Queen, lag_spatial
from esda.moran import Moran, Moran_Local

geo = gpd.read_file(raiz / 'data/processed/municipios_inegi.geojson')[['CVEGEO', 'geometry']].rename(columns={'CVEGEO': 'cvegeo'})
desv = panel.filter(~pl.col('sin_casilla')).pivot(on='AELEC', index='cvegeo', values='desviacion').select('cvegeo', d2018='2018', d2021='2021', d2024='2024').drop_nulls()
mapa = geo.merge(desv.to_pandas(), on='cvegeo', how='inner').reset_index(drop=True)
W = Queen.from_dataframe(mapa, use_index=False, silence_warnings=True); W.transform = 'r'
filas = []
for c in ['d2018', 'd2021', 'd2024']:
    m = Moran(mapa[c].values, W, permutations=999)
    filas.append({'Elección': c[1:], 'Índice de Moran': round(m.I, 3), 'Valor p (permutaciones)': round(m.p_sim, 3)})
lisa = Moran_Local(mapa['d2024'].values, W, permutations=999, seed=2027)
cuadrante = np.where(lisa.p_sim < 0.05, lisa.q, 0)
mapa['zona_2024'] = pd_map = [{0: 'no significativa', 1: 'caliente (alto-alto)', 2: 'bajo rodeado de alto', 3: 'fría (bajo-bajo)', 4: 'alto rodeado de bajo'}[q] for q in cuadrante]
zonas = pl.DataFrame({'cvegeo': mapa['cvegeo'].tolist(), 'zona_participacion_2024': mapa['zona_2024'].tolist()})
print(pl.DataFrame(filas))
zonas.join(panel.filter(pl.col('AELEC') == 2024).select('cvegeo', 'estado'), on='cvegeo').group_by('zona_participacion_2024').agg(
    pl.len().alias('Municipios'), pl.col('estado').value_counts(sort=True).head(4).struct.field('estado').str.join(', ').alias('Estados con más')
).sort('Municipios', descending=True)

shape: (3, 3)
┌──────────┬─────────────────┬─────────────────────────┐
│ Elección ┆ Índice de Moran ┆ Valor p (permutaciones) │
│ ---      ┆ ---             ┆ ---                     │
│ str      ┆ f64             ┆ f64                     │
╞══════════╪═════════════════╪═════════════════════════╡
│ 2018     ┆ 0.524           ┆ 0.001                   │
│ 2021     ┆ 0.524           ┆ 0.001                   │
│ 2024     ┆ 0.55            ┆ 0.001                   │
└──────────┴─────────────────┴─────────────────────────┘


zona_participacion_2024,Municipios,Estados con más
str,u32,str
"""no significativa""",1652,"""Oaxaca, Veracruz de Ignacio de la Llave, Puebla, México"""
"""fría (bajo-bajo)""",371,"""Oaxaca, Michoacán de Ocampo, Durango, Chihuahua"""
"""caliente (alto-alto)""",337,"""Yucatán, Puebla, Tlaxcala, Chiapas"""
"""alto rodeado de bajo""",43,"""Oaxaca, Michoacán de Ocampo, Guerrero, Jalisco"""
"""bajo rodeado de alto""",21,"""Oaxaca, Puebla, Chiapas, Morelos"""


**Qué vemos:**

- **La abstención está fuertemente agrupada:** el índice de Moran va de 0.52 a 0.55 en 2018, 2021 y 2024 (0 = sin agrupamiento).
- **Zonas frías de 2024:** 371 municipios de baja participación rodeados de baja participación, la mayoría en Oaxaca. Le siguen Michoacán, Durango y Chihuahua.
- **Zonas calientes:** 337 municipios, sobre todo en Yucatán, Puebla y Tlaxcala.

**Para el mapa:** una vista de **regiones** (zonas frías), no solo de municipios aislados (K19).

### Oaxaca: el efecto de elegir ayuntamiento el mismo día

In [8]:
sni = pl.read_csv(raiz / 'data/reference/oaxaca_municipios_sni.csv', infer_schema=False)['cvegeo_inegi'].to_list()
(panel.filter(pl.col('clave_estado') == 20, ~pl.col('sin_casilla'), pl.col('participacion').is_not_null())
 .with_columns(regimen=pl.when(pl.col('cvegeo').is_in(sni)).then(pl.lit('usos y costumbres')).otherwise(pl.lit('partidos')))
 .group_by('AELEC', 'regimen').agg(p=(pl.col('SV').sum() / (pl.col('SV').sum() + pl.col('NV').sum()) * 100).round(1))
 .pivot(on='regimen', index='AELEC', values='p').sort('AELEC')
 .with_columns(brecha=(pl.col('partidos') - pl.col('usos y costumbres')).round(1))
 .join(concurrencia.filter(pl.col('clave_estado') == 20, ~pl.col('es_sni')).group_by('AELEC').agg(pl.col('ayuntamiento_mismo_dia').any()), on='AELEC')
 .sort('AELEC')
 .rename({'AELEC': 'Elección', 'partidos': 'Municipios de partidos (%)', 'usos y costumbres': 'Municipios de usos y costumbres (%)',
          'brecha': 'Brecha (pp)', 'ayuntamiento_mismo_dia': '¿Los de partidos eligen ayuntamiento ese día?'}))

Elección,Municipios de usos y costumbres (%),Municipios de partidos (%),Brecha (pp),¿Los de partidos eligen ayuntamiento ese día?
i16,f64,f64,f64,bool
2009,37.7,42.4,4.7,false
2012,55.6,63.1,7.5,false
2015,39.6,41.4,1.8,false
2018,57.8,69.9,12.1,true
2021,42.1,62.8,20.7,true
2024,48.2,64.3,16.1,true


**Qué vemos:** mientras ningún municipio de Oaxaca elegía ayuntamiento el mismo día que la federal (2009 a 2015), la brecha era de 2 a 8 puntos. Desde 2018, cuando los de partidos empezaron a hacerlo, subió a 12–21 puntos.

**Interpretación:** es la evidencia más limpia que tenemos del efecto de la elección municipal, porque compara antes y después dentro de un mismo estado. No es un experimento: pudieron cambiar otras cosas. **Para 2027:** los 418 municipios de usos y costumbres no tendrán elección municipal ese día.

## 4 · Las variables de cada municipio, transición por transición

Para que la validación del modelo sea honesta, cada transición usa **solo información disponible antes de su elección intermedia**:

| Transición | Historia | Elecciones locales | Censo, CONEVAL, CONAPO | Homicidios | Margen presidencial |
|---|---|---|---|---|---|
| 2012 → 2015 | 2009 y 2012 | 2012 y 2015 | 2010 | 2012 a 2014 | 2012 |
| 2018 → 2021 | 2015 y 2018 | 2018 y 2021 | 2020 | 2018 a 2020 | 2018 |
| 2024 → 2027 | 2021 y 2024 | 2024 y 2027 | 2020 | 2022 a 2024 | 2024 |

**Por qué no usamos la Encuesta Intercensal 2015:** no tiene lengua indígena para 1,145 municipios, ni separa la migración a otro país.

In [9]:
TRANSICIONES = {2015: (2012, 2009), 2021: (2018, 2015), 2027: (2024, 2021)}
FUENTE_SOCIAL = {2015: 2010, 2021: 2020, 2027: 2020}
HOMICIDIOS = {2015: (2012, 2014), 2021: (2018, 2020), 2027: (2022, 2024)}

# --- Competitividad presidencial por municipio (votos por candidatura según los archivos oficiales de candidaturas)
ubicacion = secciones.select('AELEC', 'EDOCVE', 'SECCION', 'municipio_2024').unique()
def mapa_anio(anio):
    return (ubicacion.with_columns(d=(pl.col('AELEC') - anio).abs()).sort('d', 'AELEC')
            .unique(['EDOCVE', 'SECCION'], keep='first').select('EDOCVE', 'SECCION', 'municipio_2024'))
def leer_csv_actas(ruta):
    lineas = ruta.read_text(encoding='latin-1').split('\n')
    inicio = next(i for i, l in enumerate(lineas) if 'CLAVE_CASILLA' in l)
    return pl.read_csv(io.StringIO('\n'.join(lineas[inicio:])), separator='|', infer_schema=False, truncate_ragged_lines=True)
numero = lambda c: pl.col(c).cast(pl.Utf8).str.replace_all(r"[=\"',]", '').cast(pl.Int64, strict=False).fill_null(0)
CANDIDATURAS = {
    2012: [['votos_pan'], ['votos_pri', 'votos_pvem', 'votos_pri_pvem'],
           ['votos_prd', 'votos_pt', 'votos_mc', 'votos_prd_pt_mc', 'votos_prd_pt', 'votos_prd_mc', 'votos_pt_mc'], ['votos_nva_alianza']],
    2018: [['PT', 'MORENA', 'ENCUENTRO SOCIAL', 'PT_MORENA_PES', 'PT_MORENA', 'PT_PES', 'MORENA_PES'],
           ['PAN', 'PRD', 'MOVIMIENTO CIUDADANO', 'PAN_PRD_MC', 'PAN_PRD', 'PAN_MC', 'PRD_MC'],
           ['PRI', 'PVEM', 'NUEVA ALIANZA', 'PRI_PVEM_NA', 'PRI_PVEM', 'PRI_NA', 'PVEM_NA'], ['CAND_IND_01'], ['CAND_IND_02']],
    2024: [['PVEM', 'PT', 'MORENA', 'PVEM_PT_MORENA', 'PVEM_PT', 'PVEM_MORENA', 'PT_MORENA'],
           ['PAN', 'PRI', 'PRD', 'PAN_PRI_PRD', 'PAN_PRI', 'PAN_PRD', 'PRI_PRD'], ['MC']],
}
partes = []
for anio, grupos in CANDIDATURAS.items():
    nombres = [f'c{i}' for i in range(len(grupos))]
    if anio == 2012:
        t = pl.read_parquet(E / 'actas_2012.parquet').filter(pl.col('tipo_casilla').is_in(['B', 'C', 'E']), ~pl.col('anulada_tepjf'))
        t = t.select('EDOCVE', 'SECCION', *[pl.sum_horizontal(g).alias(n) for g, n in zip(grupos, nombres)])
    else:
        t = leer_csv_actas(raiz / 'data/raw' / {2018: 'computos_2018/PRES_2018.csv', 2024: 'computos_2024/PRES_2024.csv'}[anio])
        entidad = 'ID_ENTIDAD' if 'ID_ENTIDAD' in t.columns else 'ID_ESTADO'
        t = t.filter(pl.col('TIPO_CASILLA').is_in(['B', 'C', 'E'])).select(
            EDOCVE=numero(entidad).cast(pl.Int8), SECCION=numero('SECCION').cast(pl.Int32),
            **{n: pl.sum_horizontal([numero(c) for c in g]) for g, n in zip(grupos, nombres)})
    t = t.join(mapa_anio(anio), on=['EDOCVE', 'SECCION'], how='left')
    assert t['municipio_2024'].null_count() == 0
    m = t.group_by(pl.col('municipio_2024').alias('cvegeo')).agg(pl.col(nombres).sum())
    votos = m.select(nombres).to_numpy().astype(float); total = votos.sum(1); orden = -np.sort(-votos, 1); hay = total > 0
    partes.append(m.select('cvegeo').with_columns(
        AELEC=pl.lit(anio, pl.Int16),
        margen=pl.Series(np.where(hay, (orden[:, 0] - orden[:, 1]) / np.where(hay, total, 1) * 100, np.nan)).fill_nan(None),
        partidos_efectivos=pl.Series(np.where(hay, 1 / np.where(hay, ((votos / np.where(hay, total, 1)[:, None]) ** 2).sum(1), 1), np.nan)).fill_nan(None)))
competitividad = pl.concat(partes)

# --- Población CONAPO y cobertura de la lista (20 años o más, mismo sexo y grupo de edad)
proyecciones = pl.read_parquet(E / 'conapo_proyecciones_edad_sexo.parquet')
poblacion_total = proyecciones.group_by('cvegeo', 'anio').agg(poblacion=pl.col('poblacion').sum())
cobertura = (por_edad.filter(pl.col('grupo_edad') != '18-19')
             .join(proyecciones.rename({'anio': 'AELEC'}).with_columns(pl.col('AELEC').cast(pl.Int16)), on=['cvegeo', 'AELEC', 'sexo', 'grupo_edad'])
             .group_by('cvegeo', 'AELEC').agg(cobertura_lista_20mas=pl.col('LN').sum() / pl.col('poblacion').sum() * 100))

def vecinos(df, columna):
    g = geo.merge(df.select('cvegeo', columna).drop_nulls().to_pandas(), on='cvegeo', how='inner').reset_index(drop=True)
    w = Queen.from_dataframe(g, use_index=False, silence_warnings=True); w.transform = 'r'
    return pl.DataFrame({'cvegeo': g['cvegeo'].tolist(), f'vecinos_{columna}': lag_spatial(w, g[columna].values)})

homicidios = pl.read_parquet(E / 'inegi_homicidios_municipio.parquet').filter(pl.col('clave_presente_en_edr_anio'))
tablas = []
for T, (P, Q) in TRANSICIONES.items():
    C = FUENTE_SOCIAL[T]
    t = panel.filter(pl.col('AELEC') == P).select(
        'cvegeo', 'estado', 'municipio', 'clave_estado',
        desviacion_presidencial=pl.col('desviacion'), lista_presidencial=pl.col('LN'),
        pct_rural=pl.col('LN_rural') / pl.col('LN') * 100, pct_hombres=pl.col('LN_hombres') / pl.col('LN') * 100,
        pct_20_29=pl.col('LN_20_29') / pl.col('LN') * 100, pct_65_mas=pl.col('LN_65_mas') / pl.col('LN') * 100,
        sin_casilla_presidencial=pl.col('sin_casilla').cast(pl.Float64),
    ).with_columns(eleccion_objetivo=pl.lit(T, pl.Int16), presidencial=pl.lit(P, pl.Int16))
    a0, a1 = HOMICIDIOS[T]
    uniones = [
        panel.filter(pl.col('AELEC') == Q).select('cvegeo', desviacion_intermedia_previa=pl.col('desviacion'), lista_intermedia_previa=pl.col('LN')),
        panel.filter(pl.col('AELEC') < T).group_by('cvegeo').agg(antecedentes_sin_casilla=pl.col('sin_casilla').sum(),
                                                                 antecedentes_paquete=pl.col('origen').str.starts_with('B').fill_null(False).sum()),
        concurrencia.filter(pl.col('AELEC') == T).select('cvegeo', *[pl.col(c).cast(pl.Float64).alias(c + '_intermedia') for c in ['gobernador_mismo_dia', 'diputacion_local_mismo_dia', 'ayuntamiento_mismo_dia']], pl.col('es_sni').cast(pl.Float64)),
        concurrencia.filter(pl.col('AELEC') == P).select('cvegeo', *[pl.col(c).cast(pl.Float64).alias(c + '_presidencial') for c in ['gobernador_mismo_dia', 'diputacion_local_mismo_dia', 'ayuntamiento_mismo_dia']]),
        competitividad.filter(pl.col('AELEC') == P).select('cvegeo', margen_presidencial='margen', partidos_efectivos_presidencial='partidos_efectivos'),
        cobertura.filter(pl.col('AELEC') == P).select('cvegeo', 'cobertura_lista_20mas'),
        (homicidios.filter(pl.col('anio').is_between(a0, a1)).group_by('cvegeo').agg(h=pl.col('homicidios').sum(), n=pl.len()).filter(pl.col('n') == a1 - a0 + 1)
         .join(poblacion_total.filter(pl.col('anio') == (a0 + a1) // 2), on='cvegeo')
         .select('cvegeo', homicidios_100mil_log=(pl.col('h') / (a1 - a0 + 1) / pl.col('poblacion') * 1e5).log1p())),
        pl.read_parquet(E / f'inegi_municipal_{C}.parquet').select(
            'cvegeo', escolaridad_promedio='grado_promedio_escolaridad', pct_habla_lengua_indigena='pct_3ymas_habla_lengua_indigena',
            pct_viviendas_internet='pct_viviendas_con_internet', pct_residia_otro_pais_hace5='pct_5ymas_residia_otro_pais_hace5',
            pct_residia_otra_entidad_hace5='pct_5ymas_residia_otra_entidad_hace5', poblacion_log=pl.col('poblacion_total').log()),
        pl.read_parquet(E / 'coneval_pobreza_municipal.parquet').filter(pl.col('anio') == C).select(
            'cvegeo', pct_pobreza='pobreza_pct', pct_pobreza_extrema='pobreza_extrema_pct', pct_carencia_salud='carencia_salud_pct', pct_rezago_educativo='carencia_rezago_educativo_pct'),
        pl.read_parquet(E / 'conapo_marginacion.parquet').filter(pl.col('anio') == C).select(
            'cvegeo', marginacion=-pl.col('indice_marginacion'), pct_localidades_menos_5000='pct_poblacion_localidades_menos_5000'),
        pl.read_parquet(E / 'conapo_intensidad_migratoria.parquet').filter(pl.col('anio') == C).select(
            'cvegeo', pct_viviendas_emigrantes='pct_viviendas_emigrantes_eua', pct_viviendas_remesas='pct_viviendas_reciben_remesas', pct_viviendas_retorno='pct_viviendas_migrantes_retorno'),
        vecinos(t, 'desviacion_presidencial'),
    ]
    if T != 2027:
        uniones.append(panel.filter(pl.col('AELEC') == T).select('cvegeo', desviacion_objetivo=pl.col('desviacion'), sin_casilla_objetivo=pl.col('sin_casilla'),
                                                                participacion_objetivo=pl.col('participacion'), lista_objetivo=pl.col('LN')))
    for u in uniones:
        t = t.join(u, on='cvegeo', how='left')
    tablas.append(t.with_columns(
        crecimiento_lista=(pl.col('lista_presidencial') / pl.col('lista_intermedia_previa') - 1) * 100,
        antecedentes_sin_casilla=pl.col('antecedentes_sin_casilla').fill_null(0), antecedentes_paquete=pl.col('antecedentes_paquete').fill_null(0)))
variables = pl.concat(tablas, how='diagonal_relaxed').sort('eleccion_objetivo', 'cvegeo')
variables.write_parquet(raiz / 'data/processed/variables_transiciones.parquet')
print(f'Tabla de variables: {variables.height:,} filas (municipio × transición) · {len(variables.columns)} columnas')
variables.group_by('eleccion_objetivo').agg(pl.len().alias('Municipios'), pl.col('desviacion_objetivo').is_not_null().sum().alias('Con resultado conocido')).sort('eleccion_objetivo')

Tabla de variables: 7,431 filas (municipio × transición) · 49 columnas


eleccion_objetivo,Municipios,Con resultado conocido
i16,u32,u32
2015,2477,2462
2021,2477,2470
2027,2477,0


**Qué vemos:** 2,477 municipios en cada transición. Para 2015 y 2021 conocemos el resultado, con lo que se entrena y se valida el modelo. Para 2027 es lo que queremos anticipar.

**Valores faltantes:** los municipios creados después de 2010 no tienen datos del censo 2010, y algunos datos de CONEVAL faltan en municipios pequeños. El modelo los rellena con la mediana y lo documentamos en el paso 4.

## 5 · ¿Qué variables ayudan, y en las dos transiciones?

Para cada variable medimos su correlación con **lo que la historia no anticipa** (el residuo de la intermedia dado la presidencial anterior), en **las dos transiciones pasadas**. Una variable solo es confiable para 2027 si su relación es del mismo signo y distinta de cero (IC 95 %) **en las dos**.

In [10]:
CANDIDATAS = ['margen_presidencial', 'partidos_efectivos_presidencial', 'pct_rural', 'pct_hombres', 'pct_20_29', 'pct_65_mas', 'crecimiento_lista',
              'cobertura_lista_20mas', 'escolaridad_promedio', 'pct_habla_lengua_indigena', 'pct_viviendas_internet', 'pct_residia_otro_pais_hace5',
              'pct_residia_otra_entidad_hace5', 'poblacion_log', 'pct_pobreza', 'pct_pobreza_extrema', 'pct_carencia_salud', 'pct_rezago_educativo',
              'marginacion', 'pct_localidades_menos_5000', 'pct_viviendas_emigrantes', 'pct_viviendas_remesas', 'pct_viviendas_retorno',
              'homicidios_100mil_log', 'vecinos_desviacion_presidencial', 'ayuntamiento_mismo_dia_intermedia', 'gobernador_mismo_dia_intermedia', 'es_sni']
filas = []
for T in [2015, 2021]:
    d = variables.filter((pl.col('eleccion_objetivo') == T) & pl.col('desviacion_objetivo').is_not_null() & ~pl.col('sin_casilla_objetivo').fill_null(False)
                         & pl.col('desviacion_presidencial').is_not_null())
    pend, orden = np.polyfit(d['desviacion_presidencial'], d['desviacion_objetivo'], 1)
    residuo = d['desviacion_objetivo'].to_numpy() - (orden + pend * d['desviacion_presidencial'].to_numpy())
    for v in CANDIDATAS:
        rho, bajo, alto, n = spearman_ic(d[v].cast(pl.Float64).to_numpy(), residuo, veces=400)
        filas.append({'variable': v, 'T': T, 'rho': rho, 'bajo': bajo, 'alto': alto})
r = pl.DataFrame(filas).pivot(on='T', index='variable', values=['rho', 'bajo', 'alto'])
estabilidad = r.with_columns(
    estable=((pl.col('bajo_2015') > 0) & (pl.col('bajo_2021') > 0)) | ((pl.col('alto_2015') < 0) & (pl.col('alto_2021') < 0)),
).select(
    'variable',
    pl.format('{} [{}, {}]', pl.col('rho_2015').round(2), pl.col('bajo_2015').round(2), pl.col('alto_2015').round(2)).alias('2012 → 2015: ρ [IC 95 %]'),
    pl.format('{} [{}, {}]', pl.col('rho_2021').round(2), pl.col('bajo_2021').round(2), pl.col('alto_2021').round(2)).alias('2018 → 2021: ρ [IC 95 %]'),
    pl.when(pl.col('estable')).then(pl.lit('sí')).otherwise(pl.lit('no')).alias('¿Estable en las dos?'),
    ((pl.col('rho_2015').abs() + pl.col('rho_2021').abs()) / 2).alias('fuerza'),
).sort('¿Estable en las dos?', 'fuerza', descending=[True, True]).drop('fuerza')
estabilidad

variable,2012 → 2015: ρ [IC 95 %],2018 → 2021: ρ [IC 95 %],¿Estable en las dos?
str,str,str,str
"""ayuntamiento_mismo_dia_intermedia""","""0.66 [0.63, 0.68]""","""0.43 [0.4, 0.47]""","""sí"""
"""gobernador_mismo_dia_intermedia""","""0.53 [0.5, 0.56]""","""0.24 [0.21, 0.28]""","""sí"""
"""pct_hombres""","""0.22 [0.18, 0.26]""","""0.29 [0.25, 0.33]""","""sí"""
"""cobertura_lista_20mas""","""0.29 [0.25, 0.32]""","""0.2 [0.16, 0.24]""","""sí"""
"""es_sni""","""-0.08 [-0.12, -0.05]""","""-0.34 [-0.37, -0.3]""","""sí"""
"""pct_rezago_educativo""","""0.22 [0.18, 0.25]""","""0.2 [0.16, 0.24]""","""sí"""
"""pct_residia_otra_entidad_hace5""","""-0.25 [-0.29, -0.21]""","""-0.13 [-0.18, -0.1]""","""sí"""
"""pct_65_mas""","""0.23 [0.19, 0.27]""","""0.13 [0.09, 0.17]""","""sí"""
"""escolaridad_promedio""","""-0.2 [-0.24, -0.16]""","""-0.16 [-0.2, -0.12]""","""sí"""


**Cómo leer la tabla:** "sí" en la última columna significa que la variable anticipó lo mismo en 2015 y en 2021: mismo signo y un intervalo que no incluye el 0.

**Qué vemos:**

- **Lo más fuerte y estable es lo que se elige localmente ese día:** ayuntamiento (0.66 y 0.43) y gobernador (0.53 y 0.24). Su fuerza baja en 2021, cuando casi todo el país eligió algo local.
- **Estables, pero más débiles:**
  - % de hombres y cobertura de la lista;
  - escolaridad, rezago educativo e internet, que dicen lo mismo con signos opuestos;
  - lo rural y el tamaño del municipio;
  - lengua indígena y usos y costumbres;
  - el crecimiento de la lista.
- **No estables:** el margen y el número efectivo de partidos de la presidencial, la migración internacional (anticipó bien 2015 y nada en 2021), la pobreza y los vecinos.

**Consecuencia para el modelo:** las variables estables pueden pesar en 2027; las inestables solo con regularización fuerte, y hay que comprobar en la validación que no empeoren la predicción. Los hallazgos detallados están en el reporte J (R8 a R13).

## 6 · Variables que dicen lo mismo

Muchas variables sociales están muy correlacionadas entre sí. Para explicar al INE basta con una por grupo, y el modelo debe estar preparado para eso (regularización).

In [11]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
sociales = ['escolaridad_promedio', 'pct_habla_lengua_indigena', 'pct_viviendas_internet', 'pct_residia_otro_pais_hace5', 'pct_residia_otra_entidad_hace5',
            'poblacion_log', 'pct_pobreza', 'pct_pobreza_extrema', 'pct_carencia_salud', 'pct_rezago_educativo', 'marginacion', 'pct_localidades_menos_5000',
            'pct_viviendas_emigrantes', 'pct_viviendas_remesas', 'pct_viviendas_retorno', 'homicidios_100mil_log', 'cobertura_lista_20mas']
d = variables.filter(pl.col('eleccion_objetivo') == 2021)
X = d.select(sociales).to_numpy().astype(float)
pares = []
for i in range(len(sociales)):
    for j in range(i + 1, len(sociales)):
        ok = ~np.isnan(X[:, i]) & ~np.isnan(X[:, j])
        rho = spearmanr(X[ok, i], X[ok, j]).statistic
        if abs(rho) >= 0.8:
            pares.append({'Variable 1': sociales[i], 'Variable 2': sociales[j], 'ρ': round(rho, 2)})
print(f'Pares con |ρ| ≥ 0.8: {len(pares)}')
print(pl.DataFrame(pares).sort(pl.col('ρ').abs(), descending=True))
Z = make_pipeline(SimpleImputer(strategy='median'), StandardScaler()).fit_transform(X)
pca = PCA().fit(Z)
pl.DataFrame([
    {'Componente': f'CP{k + 1}', 'Varianza explicada (%)': round(pca.explained_variance_ratio_[k] * 100, 1),
     'Variables que más pesan': ', '.join(n for n, _ in sorted(zip(sociales, pca.components_[k]), key=lambda t: -abs(t[1]))[:4])}
    for k in range(4)
])

Pares con |ρ| ≥ 0.8: 11
shape: (11, 3)
┌─────────────────────────────┬───────────────────────┬───────┐
│ Variable 1                  ┆ Variable 2            ┆ ρ     │
│ ---                         ┆ ---                   ┆ ---   │
│ str                         ┆ str                   ┆ f64   │
╞═════════════════════════════╪═══════════════════════╪═══════╡
│ escolaridad_promedio        ┆ pct_rezago_educativo  ┆ -0.94 │
│ pct_pobreza                 ┆ pct_pobreza_extrema   ┆ 0.93  │
│ pct_residia_otro_pais_hace5 ┆ pct_viviendas_retorno ┆ 0.9   │
│ escolaridad_promedio        ┆ marginacion           ┆ -0.86 │
│ pct_pobreza_extrema         ┆ marginacion           ┆ 0.86  │
│ pct_viviendas_emigrantes    ┆ pct_viviendas_remesas ┆ 0.86  │
│ pct_viviendas_remesas       ┆ pct_viviendas_retorno ┆ 0.86  │
│ pct_residia_otro_pais_hace5 ┆ pct_viviendas_remesas ┆ 0.82  │
│ pct_viviendas_internet      ┆ marginacion           ┆ -0.81 │
│ pct_pobreza                 ┆ marginacion           ┆ 0.81  │
│

Componente,Varianza explicada (%),Variables que más pesan
str,f64,str
"""CP1""",36.4,"""escolaridad_promedio, marginacion, pct_rezago_educativo, pct_viviendas_internet"""
"""CP2""",20.1,"""pct_viviendas_retorno, pct_residia_otro_pais_hace5, pct_viviendas_remesas, pct_viviendas_e…"
"""CP3""",9.2,"""pct_carencia_salud, poblacion_log, cobertura_lista_20mas, pct_localidades_menos_5000"""
"""CP4""",7.2,"""homicidios_100mil_log, pct_residia_otra_entidad_hace5, cobertura_lista_20mas, poblacion_lo…"


**Qué vemos:**

- **El primer componente es "desarrollo social":** escolaridad de un lado; pobreza, marginación y rezago del otro.
- **El segundo es "migración internacional".**
- **Los cuatro primeros componentes juntan el 73 % de la información**; el resto se reparte entre muchos. Reducir todo a 2 o 3 índices perdería información. Es mejor usar todas las variables en un modelo regularizado (ridge o Huber), que reparte el peso entre las que dicen lo mismo (paso 4).
- **11 pares de variables tienen |ρ| ≥ 0.8**, como escolaridad con rezago educativo (−0.94) o pobreza con pobreza extrema (0.93).

## 7 · Hipótesis que los datos no sostuvieron

Las escribimos antes de ver los datos (reporte J) y las reportamos tal cual.

In [12]:
# ¿La violencia anticipa dónde faltarán casillas? (AUC: 0.5 = como adivinar)
filas = []
pob = poblacion_total
for T, (a0, a1) in [(2021, (2018, 2020)), (2024, (2021, 2023))]:
    hv = (homicidios.filter(pl.col('anio').is_between(a0, a1)).group_by('cvegeo').agg(h=pl.col('homicidios').sum(), n=pl.len()).filter(pl.col('n') == a1 - a0 + 1)
          .join(pob.filter(pl.col('anio') == (a0 + a1) // 2), on='cvegeo').select('cvegeo', tasa=pl.col('h') / (a1 - a0 + 1) / pl.col('poblacion') * 1e5))
    previos = panel.filter(pl.col('AELEC') < T).group_by('cvegeo').agg(antecedente=pl.col('sin_casilla').any())
    d = panel.filter(pl.col('AELEC') == T).select('cvegeo', 'sin_casilla').join(hv, on='cvegeo').join(previos, on='cvegeo', how='left').drop_nulls('tasa')
    y = d['sin_casilla'].to_numpy(); x = d['tasa'].to_numpy()
    u = mannwhitneyu(x[y], x[~y]); ant = d['antecedente'].fill_null(False).to_numpy()
    filas.append({'Elección': T, 'Municipios con lista sin casilla': int(y.sum()),
                  'AUC de la tasa de homicidios': round(u.statistic / (y.sum() * (~y).sum()), 2), 'Valor p': round(u.pvalue, 3),
                  '% que repitió, entre los que tenían antecedente': round((ant & y).sum() / max(ant.sum(), 1) * 100, 1),
                  '% de los demás': round(((~ant) & y).sum() / (~ant).sum() * 100, 1)})
pl.DataFrame(filas)

Elección,Municipios con lista sin casilla,AUC de la tasa de homicidios,Valor p,"% que repitió, entre los que tenían antecedente",% de los demás
i64,i64,f64,f64,f64,f64
2021,24,0.58,0.179,20.8,0.5
2024,18,0.49,0.921,16.2,0.3


**Qué vemos:**

- **La violencia municipal no anticipa dónde faltan casillas:** AUC de 0.58 en 2021 y 0.49 en 2024, ninguno significativo. Muchos casos son de organización comunitaria, no de violencia.
- **El antecedente sí:** de los municipios que ya habían tenido parte de su lista sin casilla, repitió el 20.8 % en 2021 y el 16.2 % en 2024, contra el 0.5 % y el 0.3 % de los demás. Eso respalda la alerta D12.

**Otras hipótesis que no se sostuvieron** (detalle en el reporte J):

- **La pobreza baja la participación:** no, los municipios más pobres votan un poco más que el país (R8).
- **La migración baja la participación más de lo que la historia anticipa:** la migración sí va con menos participación (−0.27), pero eso ya está en la historia (R10).
- **La lista inflada por migración fabrica abstención:** la lista supera a la población residente en el municipio típico (112 %), pero eso no va con menos participación. Mezcla emigración, registros antiguos y error de proyección (R10).
- **El margen de la presidencial anticipa la intermedia:** pasó en 2018 → 2021, pero no en 2012 → 2015 (R13).

## 8 · ¿Quién cuida el municipio y qué tan completo es el dato de violencia?

**Por qué lo vemos.** Una pregunta natural es si en los municipios donde el Estado está menos presente se vota menos, y si el dato de violencia de esos lugares es confiable. Usamos dos fuentes oficiales del INEGI:

- **Censo Nacional de Gobiernos Municipales (CNGMD) 2025**, con datos de 2024: quién ejerce la seguridad pública en cada municipio, cuántos policías tiene y si hay juzgado cívico. Es lo que reporta cada gobierno municipal.
- **ENVIPE 2026**, con datos de 2025: qué porcentaje de los delitos no se denuncia o no llega a una carpeta de investigación (la "cifra oculta"). **Solo existe por estado.**

**Qué esperamos.** Si la ausencia de policía fuera una señal de abandono, esos municipios votarían menos que los demás.

**Advertencia antes de empezar.** El CNGMD describe 2024. Usarlo para explicar 2015 o 2021 sería usar información del futuro. Por eso **no entra al modelo**: es una capa de contexto para el mapa.

In [13]:
seguridad = pl.read_parquet(E / 'cngmd_seguridad_municipal.parquet')
participacion = (panel.filter(pl.col('AELEC') == 2024).select('cvegeo', 'es_sni', lista_2024='LN', desviacion_2024='desviacion')
                 .join(panel.filter(pl.col('AELEC') == 2021).select('cvegeo', desviacion_2021='desviacion', ayuntamiento_mismo_dia_2021='ayuntamiento_mismo_dia'), on='cvegeo', how='left')
                 .join(panel.filter(pl.col('AELEC') == 2018).select('cvegeo', desviacion_2018='desviacion'), on='cvegeo', how='left'))
seg = seguridad.join(participacion, on='cvegeo', how='left')
print(f'Municipios en el CNGMD: {seguridad.height:,} · sin policía municipal: {(seguridad["tiene_policia_municipal"] == False).sum()} · '
      f'policías municipales (con titular): {seguridad["policias_municipales_con_titular"].sum():,}')
(seg.group_by('quien_ejerce_seguridad').agg(
    pl.len().alias('Municipios'),
    pl.col('lista_2024').sum().alias('Personas en la lista 2024'),
    pl.col('es_sni').fill_null(False).sum().alias('De usos y costumbres'),
    pl.col('desviacion_2021').median().round(1).alias('Desviación mediana 2021 (pp)'),
    pl.col('desviacion_2024').median().round(1).alias('Desviación mediana 2024 (pp)'),
).sort('Municipios', descending=True).rename({'quien_ejerce_seguridad': 'Quién ejerce la seguridad (INEGI, 2024)'}))

Municipios en el CNGMD: 2,478 · sin policía municipal: 517 · policías municipales (con titular): 166,090


"Quién ejerce la seguridad (INEGI, 2024)",Municipios,Personas en la lista 2024,De usos y costumbres,Desviación mediana 2021 (pp),Desviación mediana 2024 (pp)
str,u32,i64,u32,f64,f64
"""institución municipal""",1573,68762885,22,12.8,4.6
"""sin policía municipal: cuerpos comunales adscritos al municipio""",400,888759,360,-6.4,-7.6
"""institución municipal junto con otras o con cuerpos comunales""",324,15864109,18,6.8,2.1
"""sin policía municipal: la ejerce otra institución (estatal o federal)""",91,3949013,3,12.1,1.9
"""no identificado""",45,446769,8,17.1,6.9
"""sin policía municipal: ninguna institución la ejerce""",23,429068,3,17.1,4.1
"""no aplica (alcaldía de la Ciudad de México)""",16,7918590,0,1.1,10.1
"""sin policía municipal: cuerpos comunales autogestivos""",3,17268,2,-14.9,-15.3
"""otra situación""",2,18967,1,0.6,-5.0


In [14]:
# Sin los municipios de usos y costumbres: ¿votan menos los que no tienen policía municipal?
# En 2021 todos los que no tienen policía eligieron ayuntamiento ese día, así que la comparación justa es solo entre esos municipios
sin_sni = seg.filter(~pl.col('es_sni').fill_null(False), pl.col('tiene_policia_municipal').is_not_null())
con, sin = (sin_sni.filter(pl.col('tiene_policia_municipal') == v) for v in (True, False))
print('Municipios sin policía municipal que eligieron ayuntamiento el mismo día en 2021:',
      f'{sin["ayuntamiento_mismo_dia_2021"].sum()} de {sin.height}')
filas = []
for nombre, valor, solo_ayuntamiento in [
        ('Desviación 2024 (presidencial)', pl.col('desviacion_2024'), False),
        ('Desviación 2021 (intermedia), todos', pl.col('desviacion_2021'), False),
        ('Desviación 2021, solo los que eligieron ayuntamiento ese día', pl.col('desviacion_2021'), True),
        ('Cambio de 2018 a 2021, solo los que eligieron ayuntamiento ese día', pl.col('desviacion_2021') - pl.col('desviacion_2018'), True)]:
    a, b = (g.filter(pl.col('ayuntamiento_mismo_dia_2021').fill_null(False) | (not solo_ayuntamiento)).select(valor).to_series().drop_nulls().to_numpy() for g in (con, sin))
    filas.append({'Medida (pp)': nombre, 'Con policía municipal (mediana)': round(float(np.median(a)), 1),
                  'Sin policía municipal (mediana)': round(float(np.median(b)), 1),
                  'Municipios con / sin': f'{len(a):,} / {len(b)}', 'Valor p (Mann-Whitney)': round(mannwhitneyu(a, b).pvalue, 3)})
pl.DataFrame(filas)

Municipios sin policía municipal que eligieron ayuntamiento el mismo día en 2021: 149 de 149


Medida (pp),Con policía municipal (mediana),Sin policía municipal (mediana),Municipios con / sin,Valor p (Mann-Whitney)
str,f64,f64,str,f64
"""Desviación 2024 (presidencial)""",4.5,4.3,"""1,855 / 148""",0.982
"""Desviación 2021 (intermedia), todos""",12.1,16.9,"""1,850 / 149""",0.008
"""Desviación 2021, solo los que eligieron ayuntamiento ese día""",13.2,16.9,"""1,729 / 149""",0.092
"""Cambio de 2018 a 2021, solo los que eligieron ayuntamiento ese día""",6.5,7.5,"""1,721 / 149""",0.068


In [15]:
# Los que no tienen policía municipal fuera de usos y costumbres, por estado
(sin.group_by('cve_ent').agg(pl.len().alias('Municipios sin policía municipal'),
                             pl.col('quien_ejerce_seguridad').mode().first().alias('Situación más común'),
                             pl.col('desviacion_2021').median().round(1).alias('Desviación mediana 2021 (pp)'))
 .join(panel.select(pl.col('cvegeo').str.slice(0, 2).alias('cve_ent'), 'estado').unique(), on='cve_ent')
 .sort('Municipios sin policía municipal', descending=True).head(8).drop('cve_ent'))

Municipios sin policía municipal,Situación más común,Desviación mediana 2021 (pp),estado
u32,str,f64,str
43,"""sin policía municipal: cuerpos comunales adscritos al municipio""",18.0,"""Oaxaca"""
43,"""sin policía municipal: la ejerce otra institución (estatal o federal)""",18.1,"""Tamaulipas"""
15,"""sin policía municipal: la ejerce otra institución (estatal o federal)""",15.7,"""Veracruz de Ignacio de la Llave"""
10,"""sin policía municipal: la ejerce otra institución (estatal o federal)""",6.4,"""Chihuahua"""
7,"""sin policía municipal: la ejerce otra institución (estatal o federal)""",5.6,"""Guanajuato"""
6,"""sin policía municipal: ninguna institución la ejerce""",20.1,"""Nuevo León"""
5,"""sin policía municipal: ninguna institución la ejerce""",12.7,"""Guerrero"""
4,"""sin policía municipal: la ejerce otra institución (estatal o federal)""",25.9,"""Coahuila de Zaragoza"""


**Qué vemos:**

- **517 municipios no tenían policía propia en 2024.** La función la ejercían cuerpos comunales adscritos al municipio (400, todos en Oaxaca), otra institución estatal o federal (91; en Tamaulipas, sus 43 municipios) o nadie (23).
- **Los 400 municipios con cuerpos comunales votan menos** (−7.6 pp en 2024), pero **360 son de usos y costumbres**, que ya sabemos que votan menos en elecciones federales (sección 4). No es la seguridad lo que los distingue.
- **Fuera de usos y costumbres, no tener policía municipal no va con votar menos.**
  - En 2024 votaron igual que los que sí tienen (4.3 contra 4.5 pp sobre el país).
  - En 2021 votaron más (16.9 contra 12.1), pero es porque **todos** eligieron ayuntamiento ese día; 43 son de Tamaulipas. Comparados solo con los municipios que también lo eligieron, la diferencia es menor (16.9 contra 13.2) y no es significativa (p = 0.09); su cambio de 2018 a 2021 también es parecido (7.5 contra 6.5, p = 0.07).
- **"Ninguna institución la ejerce" (23 municipios) no es lo mismo que "zona sin Estado".** Es lo que reportó el gobierno municipal. Incluye, por ejemplo, a la capital de Campeche, que también aparece así en el tabulado oficial del INEGI. Hay que mostrarlo como lo dice el INEGI y no reinterpretarlo.

**Cómo usarlo en el mapa:** como **capa de contexto**, con las categorías del INEGI tal cual. **No como alerta**, porque no anticipa menor participación.

In [16]:
# ¿Qué tan completo es el dato de violencia en cada estado?
# Homicidios del INEGI (certificados de defunción, por entidad donde ocurrió) contra víctimas de homicidio doloso y feminicidio del SESNSP (carpetas de las fiscalías)
ENTIDADES = [f'{i:02d}' for i in range(1, 33)]
hom = pl.read_parquet(E / 'inegi_homicidios_municipio.parquet')
hom_sin_municipio = pl.read_parquet(E / 'inegi_homicidios_no_especificado.parquet')
inegi_estado = (pl.concat([hom.select(pl.col('cvegeo').str.slice(0, 2).alias('cve_ent'), 'anio', 'homicidios'),
                           hom_sin_municipio.filter(pl.col('cve_ent_ocurrencia').is_in(ENTIDADES)).select(pl.col('cve_ent_ocurrencia').alias('cve_ent'), 'anio', 'homicidios')])
                .filter(pl.col('cve_ent').is_in(ENTIDADES)).group_by('cve_ent', 'anio').agg(inegi=pl.col('homicidios').sum()))
sesnsp_estado = (pl.read_parquet(E / 'sesnsp_victimas_estatal.parquet').filter(pl.col('delito').is_in(['homicidio_doloso', 'feminicidio']))
                 .group_by('cve_ent', 'anio').agg(sesnsp=pl.col('victimas').sum()))
fuentes = inegi_estado.join(sesnsp_estado, on=['cve_ent', 'anio']).filter(pl.col('anio').is_between(2019, 2024)).with_columns(razon=pl.col('inegi') / pl.col('sesnsp'))
print(fuentes.group_by('anio').agg(pl.col('inegi').sum(), pl.col('sesnsp').sum()).with_columns((pl.col('inegi') / pl.col('sesnsp')).round(3).alias('INEGI / SESNSP'))
      .sort('anio').rename({'anio': 'Año', 'inegi': 'Homicidios INEGI', 'sesnsp': 'Víctimas SESNSP'}))

envipe = pl.read_parquet(E / 'envipe_cifra_oculta_estatal.parquet').filter(pl.col('cve_ent') != '00')
comparacion_estados = (fuentes.group_by('cve_ent').agg(
        pl.col('razon').min().alias('minima'), pl.col('razon').max().alias('maxima'), pl.col('razon').mean().alias('promedio'))
    # Regla propuesta (D17): una fuente "cuenta más" si en al menos 5 de los 6 años supera a la otra por más de 5 %
    .join(fuentes.group_by('cve_ent').agg(anios_inegi_mas=(pl.col('razon') > 1.05).sum(), anios_sesnsp_mas=(pl.col('razon') < 1 / 1.05).sum()), on='cve_ent')
    .with_columns(comparacion_homicidios_inegi_sesnsp=pl.when(pl.col('anios_inegi_mas') >= 5).then(pl.lit('el INEGI cuenta más homicidios'))
                  .when(pl.col('anios_sesnsp_mas') >= 5).then(pl.lit('el SESNSP cuenta más homicidios')).otherwise(pl.lit('las dos fuentes coinciden')))
    .join(envipe.select('cve_ent', 'entidad', cifra_oculta_estatal_2025='cifra_oculta_pct_2025'), on='cve_ent'))
(comparacion_estados.sort('promedio', descending=True).select(
    pl.col('entidad').alias('Estado'),
    pl.col('promedio').round(2).alias('INEGI / SESNSP, promedio 2019-2024'),
    pl.format('{} a {}', pl.col('minima').round(2), pl.col('maxima').round(2)).alias('Rango'),
    pl.format('{} / {}', pl.col('anios_inegi_mas'), pl.col('anios_sesnsp_mas')).alias('Años en que cuenta 5 % más: INEGI / SESNSP'),
    pl.col('comparacion_homicidios_inegi_sesnsp').alias('Lectura'),
    pl.col('cifra_oculta_estatal_2025').round(1).alias('Cifra oculta ENVIPE 2025 (%)')))

shape: (6, 4)
┌──────┬──────────────────┬─────────────────┬────────────────┐
│ Año  ┆ Homicidios INEGI ┆ Víctimas SESNSP ┆ INEGI / SESNSP │
│ ---  ┆ ---              ┆ ---             ┆ ---            │
│ i32  ┆ i64              ┆ i64             ┆ f64            │
╞══════╪══════════════════╪═════════════════╪════════════════╡
│ 2019 ┆ 35527            ┆ 35694           ┆ 0.995          │
│ 2020 ┆ 36285            ┆ 35548           ┆ 1.021          │
│ 2021 ┆ 34733            ┆ 34380           ┆ 1.01           │
│ 2022 ┆ 32435            ┆ 31971           ┆ 1.015          │
│ 2023 ┆ 30943            ┆ 30593           ┆ 1.011          │
│ 2024 ┆ 31902            ┆ 30918           ┆ 1.032          │
└──────┴──────────────────┴─────────────────┴────────────────┘


Estado,"INEGI / SESNSP, promedio 2019-2024",Rango,Años en que cuenta 5 % más: INEGI / SESNSP,Lectura,Cifra oculta ENVIPE 2025 (%)
str,f64,str,str,str,f64
"""Guanajuato""",1.19,"""1.13 a 1.27""","""6 / 0""","""el INEGI cuenta más homicidios""",92.3
"""Yucatán""",1.17,"""1.05 a 1.25""","""5 / 0""","""el INEGI cuenta más homicidios""",91.0
"""Hidalgo""",1.17,"""0.96 a 1.39""","""5 / 0""","""el INEGI cuenta más homicidios""",92.7
"""México""",1.15,"""1.04 a 1.37""","""5 / 0""","""el INEGI cuenta más homicidios""",94.0
"""Tlaxcala""",1.1,"""1.02 a 1.19""","""5 / 0""","""el INEGI cuenta más homicidios""",91.5
"""Chihuahua""",1.08,"""0.99 a 1.26""","""3 / 0""","""las dos fuentes coinciden""",94.0
"""Sinaloa""",1.08,"""0.96 a 1.18""","""4 / 0""","""las dos fuentes coinciden""",94.8
"""Zacatecas""",1.06,"""1.0 a 1.14""","""4 / 0""","""las dos fuentes coinciden""",92.6
"""San Luis Potosí""",1.05,"""0.96 a 1.13""","""4 / 0""","""las dos fuentes coinciden""",93.2


**Qué vemos:**

- **A nivel nacional, las dos fuentes casi coinciden de 2019 a 2024:** el INEGI cuenta entre 0.5 % menos y 3 % más homicidios que el SESNSP.
  - La brecha de 15 % que se reporta para 2025 usa las cifras **preliminares** del INEGI (defunciones **registradas** en 2025). Aquí contamos por año y lugar en que **ocurrió** la defunción, y 2025 todavía no está en los datos abiertos del INEGI.
- **Por estado sí hay diferencias que se repiten.** Llamamos "una fuente cuenta más" cuando, en al menos 5 de los 6 años, supera a la otra por más de 5 %:
  - **El INEGI cuenta más en Guanajuato, Yucatán, Hidalgo, Estado de México y Tlaxcala.** Ahí el SESNSP queda corto. En Guanajuato pasa los 6 años, por 13 % a 27 %.
  - **El SESNSP cuenta más en Veracruz, Oaxaca, Jalisco, Coahuila y Durango.** Puede ser porque el INEGI asigna la defunción al lugar donde ocurrió y el SESNSP al lugar del delito, o porque las fiscalías registran víctimas que no tienen certificado de defunción en ese estado. Con estos datos no podemos saber cuál de las dos.
  - En los otros 22 estados las dos fuentes coinciden.
- **La cifra oculta de la ENVIPE es alta en todos los estados** (de 88.4 % a 96.3 %). Mide los delitos comunes que no se denuncian, **no los homicidios**, y no existe por municipio.

**Decisión que proponemos (D17, para el paso 5):** la capa de violencia del mapa usa los **homicidios del INEGI por municipio**, que no dependen de la denuncia. La etiqueta de cada estado ("el INEGI cuenta más", "el SESNSP cuenta más" o "las dos fuentes coinciden") va como aviso de qué tan completo es el dato. Los delitos que solo da el SESNSP (extorsión, secuestro) van con ese aviso.

**Lo que no hacemos:** un índice de "control criminal" o de "zonas sin Estado". No hay un dato público que lo sostenga, y el INE reserva por 5 años el municipio de las secciones que atiende de manera especial por inseguridad.

## 9 · Indicadores por municipio para el mapa

Guardamos en `data/processed/indicadores_mapa.parquet` los indicadores municipales que salen de este notebook. Son vistas y alertas candidatas (reporte K), que se combinarán con el riesgo del modelo en el paso 5.

In [17]:
persistente = panel.filter(pl.col('desviacion').is_not_null()).group_by('cvegeo').agg(
    elecciones_por_debajo_del_pais=(pl.col('desviacion') < 0).sum(), elecciones_con_dato=pl.len())
antecedentes = panel.group_by('cvegeo').agg(
    elecciones_con_lista_sin_casilla=pl.col('sin_casilla').sum(), elecciones_con_paquete_perdido=pl.col('origen').str.starts_with('B').fill_null(False).sum(),
    ultima_eleccion_sin_casilla=pl.col('AELEC').filter(pl.col('sin_casilla')).max())
jovenes = panel.filter(pl.col('AELEC') == 2024).select(
    'cvegeo', 'estado', 'municipio',
    brecha_jovenes_2024=(pl.col('SV_20_29') / (pl.col('SV_20_29') + pl.col('NV_20_29')) - pl.col('SV') / (pl.col('SV') + pl.col('NV'))) * 100,
    lista_20_29_2024=pl.col('LN_20_29'))
indicadores = (jovenes.join(persistente, on='cvegeo', how='left').join(antecedentes, on='cvegeo', how='left')
               .join(zonas, on='cvegeo', how='left')
               .join(credencial.select('cvegeo', credencializacion_18_19_2021='credencializacion_18_19'), on='cvegeo', how='left')
               .join(concurrencia.filter(pl.col('AELEC') == 2027).select('cvegeo', 'es_sni', ayuntamiento_mismo_dia_2027='ayuntamiento_mismo_dia', gobernador_mismo_dia_2027='gobernador_mismo_dia'), on='cvegeo', how='left')
               .with_columns(alerta_sin_casilla=pl.when(pl.col('elecciones_con_lista_sin_casilla') >= 2).then(pl.lit('recurrente'))
                             .when(pl.col('elecciones_con_lista_sin_casilla') == 1).then(pl.lit('antecedente')).otherwise(pl.lit('sin antecedente')))
               # Contexto de seguridad (CNGMD 2025 por municipio; comparación de fuentes y ENVIPE por estado). No son alertas.
               .join(seguridad.join(poblacion_total.filter(pl.col('anio') == 2024), on='cvegeo', how='left').select(
                         'cvegeo', 'quien_ejerce_seguridad', 'tiene_policia_municipal', 'justicia_civica',
                         policias_municipales_por_10mil_habitantes=(pl.col('policias_municipales_con_titular') / pl.col('poblacion') * 1e4).round(1)),
                     on='cvegeo', how='left')
               .with_columns(cve_ent=pl.col('cvegeo').str.slice(0, 2))
               .join(comparacion_estados.select('cve_ent', 'comparacion_homicidios_inegi_sesnsp', 'cifra_oculta_estatal_2025'), on='cve_ent', how='left')
               .drop('cve_ent'))
indicadores.write_parquet(raiz / 'data/processed/indicadores_mapa.parquet')
print(f'Indicadores guardados: {indicadores.height:,} municipios')
indicadores.select(
    (pl.col('elecciones_por_debajo_del_pais') == 6).sum().alias('Por debajo del país en las 6 elecciones'),
    (pl.col('alerta_sin_casilla') == 'recurrente').sum().alias('Alerta "sin casilla" recurrente'),
    (pl.col('alerta_sin_casilla') == 'antecedente').sum().alias('Alerta "sin casilla" con antecedente'),
    (pl.col('zona_participacion_2024') == 'fría (bajo-bajo)').sum().alias('En zona fría 2024'),
    (~pl.col('ayuntamiento_mismo_dia_2027')).sum().alias('Sin elección municipal el mismo día en 2027'),
    (pl.col('tiene_policia_municipal') == False).sum().alias('Sin policía municipal (INEGI 2024)'),
)

Indicadores guardados: 2,477 municipios


Por debajo del país en las 6 elecciones,"Alerta ""sin casilla"" recurrente","Alerta ""sin casilla"" con antecedente",En zona fría 2024,Sin elección municipal el mismo día en 2027,Sin policía municipal (INEGI 2024)
u32,u32,u32,u32,u32,u32
285,16,59,371,669,517


## Siguiente paso

**Paso 4 · Modelo y validación:** con `data/processed/variables_transiciones.parquet`, aprender de una transición y predecir la otra (D15), comparar contra la regla simple justa y producir el riesgo de 2027 con su incertidumbre.